# But First, Coffee: Competitive Revenue Intelligence

## 01 — Data Collection

This notebook documents, loads, and validates the raw datasets collected for the competitive revenue analysis of:

- But First, Coffee (BFC)
- Starbucks Philippines
- The Coffee Bean & Tea Leaf Philippines (CBTL)

The raw data cover:

- Menu and pricing
- Customer reviews
- Individual customer ratings
- Store footprint
- Menu breadth
- Customer access channels
- Partnerships
- Official menu information
- Official company capabilities

### Purpose

The purpose of this notebook is to:

1. Document the data-collection methodology.
2. Load the raw master workbook.
3. Inventory the available datasets.
4. Validate record counts and structure.
5. Identify remaining collection limitations.
6. Preserve an auditable raw-data baseline before cleaning.

> No cleaning, imputation, standardization, outlier removal, or transformation will be performed in this notebook.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"

RAW_FILE = DATA_RAW / "BFC_competitive_data_collection_v7_individual_ratings.xlsx"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data folder: {DATA_RAW}")
print(f"Raw workbook: {RAW_FILE.name}")
print(f"File exists: {RAW_FILE.exists()}")

Project root: /Users/jannoelvero/Desktop/but_first_coffee_revenue_analysis
Raw data folder: /Users/jannoelvero/Desktop/but_first_coffee_revenue_analysis/data/raw
Raw workbook: BFC_competitive_data_collection_v7_individual_ratings.xlsx
File exists: True


In [3]:
excel_file = pd.ExcelFile(RAW_FILE)

print(f"Number of worksheets: {len(excel_file.sheet_names)}")

for i, sheet in enumerate(excel_file.sheet_names, start=1):
    print(f"{i:02d}. {sheet}")

Number of worksheets: 16
01. Collection_Log
02. Reviews_Raw
03. Menu_Prices_Raw
04. Store_Footprint
05. Channels_Partnerships
06. Menu_Breadth_Snapshots
07. Official_Portal_Menu
08. Official_Capabilities
09. Gap_Tracker
10. Review_Source_Audit
11. Company_Web_Review_Check
12. Review_Collection_Summary
13. Collection_Status_v4
14. Collection_Method
15. Individual_Ratings_Raw
16. Individual_Rating_Audit


In [4]:
inventory = []

for sheet in excel_file.sheet_names:
    df = pd.read_excel(RAW_FILE, sheet_name=sheet)

    inventory.append({
        "sheet_name": sheet,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

inventory_df = pd.DataFrame(inventory)

inventory_df

,sheet_name,rows,columns
0,Collection_Log,10,2
1,Reviews_Raw,282,9
2,Menu_Prices_Raw,150,8
3,Store_Footprint,3,7
4,Channels_Partnerships,16,6
5,Menu_Breadth_Snapshots,3,5
6,Official_Portal_Menu,36,7
7,Official_Capabilities,15,5
8,Gap_Tracker,18,3
9,Review_Source_Audit,8,10


In [5]:
reviews_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Reviews_Raw"
)

menu_prices_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Menu_Prices_Raw"
)

store_footprint_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Store_Footprint"
)

channels_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Channels_Partnerships"
)

menu_breadth_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Menu_Breadth_Snapshots"
)

official_menu_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Official_Portal_Menu"
)

official_capabilities_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Official_Capabilities"
)

individual_ratings_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Individual_Ratings_Raw"
)

print("Core analytical datasets loaded.")

Core analytical datasets loaded.


In [6]:
datasets = {
    "Reviews": reviews_raw,
    "Menu Prices": menu_prices_raw,
    "Store Footprint": store_footprint_raw,
    "Channels & Partnerships": channels_raw,
    "Menu Breadth": menu_breadth_raw,
    "Official Menu": official_menu_raw,
    "Official Capabilities": official_capabilities_raw,
    "Individual Ratings": individual_ratings_raw
}

for name, df in datasets.items():
    print(f"{name:<25} {df.shape[0]:>4} rows × {df.shape[1]:>2} columns")

Reviews                    282 rows ×  9 columns
Menu Prices                150 rows ×  8 columns
Store Footprint              3 rows ×  7 columns
Channels & Partnerships     16 rows ×  6 columns
Menu Breadth                 3 rows ×  5 columns
Official Menu               36 rows ×  7 columns
Official Capabilities       15 rows ×  5 columns
Individual Ratings          13 rows ×  9 columns


In [7]:
for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * len(name))
    
    for column in df.columns:
        print(column)


Reviews
-------
brand
branch
platform
aggregate_rating
rating_volume
review_date
review_text
initial_theme_hint
source_url

Menu Prices
-----------
brand
branch
category
product
regular_price_php
promo_price_php
platform
source_url

Store Footprint
---------------
brand
snapshot
store_count_or_statement
geographic_detail
source_quality
source_url
notes

Channels & Partnerships
-----------------------
brand
type
partner_or_channel
current_evidence
revenue_relevance
source_url

Menu Breadth
------------
brand
branch
menu_categories_visible
examples
source_url

Official Menu
-------------
brand
official_category
subcategory
product_or_offering
price_on_official_portal
commercial_note
source_url

Official Capabilities
---------------------
brand
dimension
verified_capability_or_proposition
revenue_interpretation_for_later
source_url

Individual Ratings
------------------
brand
branch
platform_origin
rating
review_recency
review_text_paraphrase
source_page
source_type
verification_note


In [8]:
review_brand_counts = (
    reviews_raw["brand"]
    .value_counts()
    .rename_axis("brand")
    .reset_index(name="review_count")
)

review_brand_counts

,brand,review_count
0,Starbucks,108
1,"But First, Coffee",104
2,The Coffee Bean & Tea Leaf,70


In [9]:
review_distribution = review_brand_counts.copy()

review_distribution["percentage"] = (
    review_distribution["review_count"]
    / review_distribution["review_count"].sum()
    * 100
).round(2)

review_distribution

,brand,review_count,percentage
0,Starbucks,108,38.30
1,"But First, Coffee",104,36.88
2,The Coffee Bean & Tea Leaf,70,24.82


In [10]:
menu_brand_counts = (
    menu_prices_raw["brand"]
    .value_counts()
    .rename_axis("brand")
    .reset_index(name="menu_observations")
)

menu_brand_counts

,brand,menu_observations
0,"But First, Coffee",50
1,Starbucks,50
2,The Coffee Bean & Tea Leaf,50


In [11]:
menu_category_counts = (
    menu_prices_raw
    .groupby(["brand", "category"])
    .size()
    .reset_index(name="observations")
    .sort_values(
        ["brand", "observations"],
        ascending=[True, False]
    )
)

menu_category_counts

,brand,category,observations
4,"But First, Coffee",Espresso,13
2,"But First, Coffee",Brewed,5
7,"But First, Coffee",Matcha,5
5,"But First, Coffee",Greater Cup,4
11,"But First, Coffee",Sip N Save,4
13,"But First, Coffee",Ube,4
1,"But First, Coffee",Auro,3
8,"But First, Coffee",Milky,3
9,"But First, Coffee",Pastry,3
0,"But First, Coffee",Add-on,2


In [12]:
branch_coverage = (
    reviews_raw
    .groupby("brand")["branch"]
    .nunique()
    .reset_index(name="unique_branches")
)

branch_coverage

,brand,unique_branches
0,"But First, Coffee",19
1,Starbucks,17
2,The Coffee Bean & Tea Leaf,19


In [13]:
reviews_by_branch = (
    reviews_raw
    .groupby(["brand", "branch"])
    .size()
    .reset_index(name="review_count")
    .sort_values(
        ["brand", "review_count"],
        ascending=[True, False]
    )
)

reviews_by_branch

,brand,branch,review_count
0,"But First, Coffee",Avida Cityflex Tower,13
9,"But First, Coffee",Quirino Highway,10
7,"But First, Coffee",Malingap Street,9
11,"But First, Coffee",San Isidro Angono,8
5,"But First, Coffee",Lucky Chinatown,7
8,"But First, Coffee",Molino,7
12,"But First, Coffee",San Pablo,7
6,"But First, Coffee",Magliman San Fernando,6
1,"But First, Coffee",Boni Avenue Mandaluyong,5
10,"But First, Coffee",Robinsons La Union,5


In [14]:
reviews_raw.head()

,brand,branch,platform,aggregate_rating,rating_volume,review_date,review_text,initial_theme_hint,source_url
0,"But First, Coffee",Avida Cityflex Tower,Foodpanda,5.0,500+,2026-04-21,There are no straws provided,packaging/accessories,https://www.foodpanda.ph/restaurant/hm34/but-first-coffee-avida-cityflex-tower/reviews
1,"But First, Coffee",Avida Cityflex Tower,Foodpanda,5.0,500+,2026-04-07,Lacks espresso. Gatas lang yung nalalasahan,taste/coffee strength,https://www.foodpanda.ph/restaurant/hm34/but-first-coffee-avida-cityflex-tower/reviews
2,"But First, Coffee",Avida Cityflex Tower,Foodpanda,5.0,500+,2026-08-22,I love but first coffee i always buy coffee there but this branch not good or standard of how ma...,consistency,https://www.foodpanda.ph/restaurant/hm34/but-first-coffee-avida-cityflex-tower/reviews
3,"But First, Coffee",Avida Cityflex Tower,Foodpanda,5.0,500+,2026-08-19,"The cap was not properly sealed, and since it was made of paper, the spilled liquid was absorbed...",packaging,https://www.foodpanda.ph/restaurant/hm34/but-first-coffee-avida-cityflex-tower/reviews
4,"But First, Coffee",Avida Cityflex Tower,Foodpanda,5.0,500+,2026-05-31,lasang tubig na may konting kape. Sayang pera sa inyo,taste/value,https://www.foodpanda.ph/restaurant/hm34/but-first-coffee-avida-cityflex-tower/reviews


In [16]:
individual_ratings_raw.head(13)

,brand,branch,platform_origin,rating,review_recency,review_text_paraphrase,source_page,source_type,verification_note
0,"But First, Coffee",Lipa Tambo,Google,5,1 year ago,"Customer praised coffee, staff, ambience and suitability for quiet work.","https://www.top-rated.online/cities/Lipa/place/p/16158503/But%20First,%20Coffee%20(BFC)%20-%20Li...",Google-origin review indexed by Top-Rated.Online,Individual 5/5 visibly attached to review
1,"But First, Coffee",Lipa Tambo,Google,4,3 months ago,Customer described the coffee as delicious.,"https://www.top-rated.online/cities/Lipa/place/p/16158503/But%20First,%20Coffee%20(BFC)%20-%20Li...",Google-origin review indexed by Top-Rated.Online,Individual 4/5 visibly attached to review
2,"But First, Coffee",Lipa Tambo,Google,5,1 year ago,"Customer praised drinks, non-dairy options, staff and modern café environment.","https://www.top-rated.online/cities/Lipa/place/p/16158503/But%20First,%20Coffee%20(BFC)%20-%20Li...",Google-origin review indexed by Top-Rated.Online,Individual 5/5 visibly attached to review
3,"But First, Coffee",Lipa Tambo,Google,5,2 years ago,"Customer highlighted affordable menu, accessibility, interior and friendly staff.","https://www.top-rated.online/cities/Lipa/place/p/16158503/But%20First,%20Coffee%20(BFC)%20-%20Li...",Google-origin review indexed by Top-Rated.Online,Individual 5/5 visibly attached to review
4,"But First, Coffee",Lipa Tambo,Google,5,1 year ago,"Customer liked Vietnamese coffee, pesto pasta and the clean setting.","https://www.top-rated.online/cities/Lipa/place/p/16158503/But%20First,%20Coffee%20(BFC)%20-%20Li...",Google-origin review indexed by Top-Rated.Online,Individual 5/5 visibly attached to review
5,"But First, Coffee",Lipa Tambo,Google,5,1 month ago,Customer praised affordable coffee and usefulness of the café as a place to stay/work.,"https://www.top-rated.online/cities/Lipa/place/p/16158503/But%20First,%20Coffee%20(BFC)%20-%20Li...",Google-origin review indexed by Top-Rated.Online,Individual 5/5 visibly attached to review
6,"But First, Coffee",Lipa Tambo,Google,5,4 months ago,Customer liked Vietnamese coffee and considered it reasonably priced.,"https://www.top-rated.online/cities/Lipa/place/p/16158503/But%20First,%20Coffee%20(BFC)%20-%20Li...",Google-origin review indexed by Top-Rated.Online,Individual 5/5 visibly attached to review
7,"But First, Coffee",Lipa Tambo,Google,5,1 year ago,"Customer praised staff recognition, customization and advance pickup preparation.","https://www.top-rated.online/cities/Lipa/place/p/16158503/But%20First,%20Coffee%20(BFC)%20-%20Li...",Google-origin review indexed by Top-Rated.Online,Individual 5/5 visibly attached to review
8,"But First, Coffee",Lipa Tambo,Google,5,11 months ago,Customer gave a concise positive assessment of the coffee.,"https://www.top-rated.online/cities/Lipa/place/p/16158503/But%20First,%20Coffee%20(BFC)%20-%20Li...",Google-origin review indexed by Top-Rated.Online,Individual 5/5 visibly attached to review
9,"But First, Coffee",Lipa Tambo,Google,5,1 week ago,Customer raised a store-equipment issue affecting crew operations.,"https://www.top-rated.online/cities/Lipa/place/p/16158503/But%20First,%20Coffee%20(BFC)%20-%20Li...",Google-origin review indexed by Top-Rated.Online,Individual 5/5 visibly attached to review despite operational complaint


In [17]:
individual_rating_counts = (
    individual_ratings_raw
    .groupby("brand")
    .agg(
        observations=("rating", "count"),
        unique_branches=("branch", "nunique"),
        mean_rating=("rating", "mean"),
        median_rating=("rating", "median")
    )
    .reset_index()
)

individual_rating_counts

,brand,observations,unique_branches,mean_rating,median_rating
0,"But First, Coffee",13,2,4.692308,5.0


In [18]:
rating_frequency = (
    individual_ratings_raw["rating"]
    .value_counts()
    .sort_index()
    .rename_axis("rating")
    .reset_index(name="count")
)

rating_frequency

,rating,count
0,2,1
1,4,1
2,5,11


In [19]:
for name, df in datasets.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]
    
    print(f"\n{name}")
    print("-" * len(name))
    
    if missing.empty:
        print("No missing values detected.")
    else:
        print(missing)


Reviews
-------
review_date    249
dtype: int64

Menu Prices
-----------
promo_price_php    105
dtype: int64

Store Footprint
---------------
No missing values detected.

Channels & Partnerships
-----------------------
No missing values detected.

Menu Breadth
------------
No missing values detected.

Official Menu
-------------
No missing values detected.

Official Capabilities
---------------------
No missing values detected.

Individual Ratings
------------------
No missing values detected.


In [20]:
duplicate_summary = []

for name, df in datasets.items():
    duplicate_summary.append({
        "dataset": name,
        "rows": len(df),
        "exact_duplicate_rows": df.duplicated().sum()
    })

duplicate_summary_df = pd.DataFrame(duplicate_summary)

duplicate_summary_df

,dataset,rows,exact_duplicate_rows
0,Reviews,282,0
1,Menu Prices,150,0
2,Store Footprint,3,0
3,Channels & Partnerships,16,0
4,Menu Breadth,3,0
5,Official Menu,36,0
6,Official Capabilities,15,0
7,Individual Ratings,13,0


In [21]:
for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * len(name))
    print(df.dtypes)


Reviews
-------
brand                     str
branch                    str
platform                  str
aggregate_rating      float64
rating_volume             str
review_date               str
review_text               str
initial_theme_hint        str
source_url                str
dtype: object

Menu Prices
-----------
brand                    str
branch                   str
category                 str
product                  str
regular_price_php      int64
promo_price_php      float64
platform                 str
source_url               str
dtype: object

Store Footprint
---------------
brand                       str
snapshot                    str
store_count_or_statement    str
geographic_detail           str
source_quality              str
source_url                  str
notes                       str
dtype: object

Channels & Partnerships
-----------------------
brand                 str
type                  str
partner_or_channel    str
current_evidence      str
reve

## Initial Data Collection Audit

The raw master workbook was successfully loaded and contains 16 worksheets.

The principal analytical datasets include:

- Written customer reviews
- Menu and pricing observations
- Store-footprint information
- Channel and partnership information
- Menu-breadth information
- Official menu information
- Official company capabilities
- Individual customer ratings

### Collection Principles

The raw datasets are preserved without modification in this notebook.

No missing values have been imputed, no duplicates have been removed, no categories have been standardized, and no observations have been excluded.

Potential data-quality issues identified during this audit will be addressed systematically in `02_data_cleaning.ipynb`.

### Important Methodological Distinction

Branch-level aggregate ratings and individual customer ratings represent different units of analysis and must not be treated interchangeably.

Written customer reviews without an explicitly attached individual star rating remain suitable for customer-experience and thematic analysis, but they will not be used as individual rating observations for inferential rating comparisons.

In [22]:
theme_counts = (
    reviews_raw["initial_theme_hint"]
    .value_counts(dropna=False)
    .rename_axis("initial_theme_hint")
    .reset_index(name="count")
)

theme_counts

,initial_theme_hint,count
0,packaging,39
1,order accuracy,31
2,taste,31
3,missing item,21
4,customization,18
...,...,...
68,service/customization,1
69,portion/loyalty,1
70,consistency/positive,1
71,positive,1


In [23]:
print(f"Number of raw theme labels: {reviews_raw['initial_theme_hint'].nunique()}")

print("\nRaw themes:")
for theme in sorted(reviews_raw["initial_theme_hint"].dropna().unique()):
    print(f"- {theme}")

Number of raw theme labels: 73

Raw themes:
- add-on
- add-on/customization
- add-on/service
- add-on/value
- availability/consistency
- availability/customization/consistency
- availability/menu accuracy
- consistency
- consistency/customization
- consistency/positive
- consistency/taste
- customization
- customization/add-on
- customization/order accuracy
- customization/service
- customization/taste
- food quality
- food quality/availability
- food quality/value
- loyalty/service
- loyalty/taste
- missing item
- missing item/order accuracy
- order accuracy
- order accuracy/coffee strength
- order accuracy/customization
- order accuracy/taste
- packaging
- packaging/accessories
- packaging/order identification
- packaging/portion
- packaging/quality
- packaging/taste
- portion
- portion/customization
- portion/food
- portion/loyalty
- portion/order accuracy
- portion/packaging
- portion/taste
- portion/value
- portion/value/consistency
- positive
- positive/service
- service
- servic

## Data Collection Conclusion

The data-collection phase produced a multi-source competitive dataset covering menu pricing, customer reviews, store presence, customer access channels, menu breadth, official menu information, company capabilities, and a pilot sample of verified individual customer ratings.

The menu-pricing dataset contains an equal number of observations across the three competitor brands, while the written-review dataset provides broad multi-branch coverage but remains moderately unbalanced by brand.

The collection audit identified several issues that require treatment during data cleaning:

- Missing review dates
- Missing promotional prices where no promotion was observed
- Non-standardized product categories
- Non-standardized customer-experience themes
- Text-based rating-volume values
- Relative dates in the individual-rating dataset
- Potential business-key duplicates
- Different units of analysis between aggregate branch ratings and individual customer ratings

No raw observations were modified or removed during the data-collection audit.

The next phase will standardize and prepare analytical datasets while preserving the original raw workbook unchanged.